# EMNIST FPGA Cloud Classification — Measurement Notebook and Data

This notebook documents the controlled EMNIST Balanced FPGA-to-Google Cloud Run measurement series on the ZedBoard. It follows the MNIST measurement workflow while avoiding repetition of the established hardware and network implementation. **The CSV files were parsed from the original 40-run SDK log archive**, not reconstructed from aggregate accuracy figures.

**Scope:** 10 blocks of 100 distinct test images; 10 validation runs; 30 accepted formal runs; 3,000 formal requests. Live-camera demonstration and Cloud Run deployment materials are maintained separately.

**Data provenance:** [`EMNIST_FPGA_Cloud_SDK_Result_Logs_Complete.md`](EMNIST_FPGA_Cloud_SDK_Result_Logs_Complete.md) (preserves source-log filename and all seven original SDK CSV columns). For any analysis, use the formal records separately from validation. The CSV `global_image_index` is the 100-image block offset plus the SDK-local index. No source `latency_ticks` values were recalculated.


## 1. Source files and measurement boundaries

| File | Contents |
|---|---|
| [`EMNIST_FPGA_Cloud_Formal_Classification_Records.csv`](EMNIST_FPGA_Cloud_Formal_Classification_Records.csv) | 3,000 formal image requests, indices, predictions, SDK timing and source-log mapping |
| [`EMNIST_FPGA_Cloud_Validation_Records.csv`](EMNIST_FPGA_Cloud_Validation_Records.csv) | 1,000 validation requests (10 cold-first-request trials) |
| [`EMNIST_FPGA_Cloud_Run_Summaries.csv`](EMNIST_FPGA_Cloud_Run_Summaries.csv) | 40 source-reported run durations and per-run accuracy |
| [`EMNIST_FPGA_Cloud_SDK_Result_Logs_Complete.md`](EMNIST_FPGA_Cloud_SDK_Result_Logs_Complete.md) | Human-readable original measurement record and context |

Latency starts during an established TCP/TLS connection and covers the SDK-side HTTPS request/response interval; it is **not** isolated cloud inference time. Throughput uses total completed requests divided by the sum of **application-reported run durations**, not reciprocal mean latency. Validation first-request delays are excluded from formal metrics. No retries or failures outside the accepted runs are included in the formal rate.


In [ ]:
from pathlib import Path
import csv, statistics
from collections import defaultdict

base = Path('.')

def read_csv(name):
    with (base / name).open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

formal = read_csv('EMNIST_FPGA_Cloud_Formal_Classification_Records.csv')
validation = read_csv('EMNIST_FPGA_Cloud_Validation_Records.csv')
runs = read_csv('EMNIST_FPGA_Cloud_Run_Summaries.csv')

for row in formal + validation:
    for k in ('block_start','run','global_image_index','local_image_index','expected','predicted','correct','error','latency_us','latency_ticks'):
        row[k] = int(row[k])
for row in runs:
    for k in ('block_start','block_end','run','images','correct','errors'):
        row[k] = int(row[k])
    for k in ('run_duration_s','throughput_images_per_s'):
        row[k] = float(row[k])

assert len(formal) == 3000 and len(validation) == 1000
assert len([r for r in runs if r['phase']=='formal']) == 30
assert len([r for r in runs if r['phase']=='validation']) == 10
assert set(r['global_image_index'] for r in formal) == set(range(1000))
assert all(sum(r['global_image_index']==i for r in formal)==3 for i in range(1000))
assert all(r['error']==0 and r['correct']==int(r['expected']==r['predicted']) for r in formal)
print('Source records validated: 3,000 formal + 1,000 validation.')


## 2. Accuracy and communication completion

Accuracy is calculated from the **1,000 distinct images**. Each image was presented three times during the 30 accepted formal runs; these repetitions do not increase the number of independent test examples. Zero errors below refers only to completed accepted runs, not every attempted startup or TCP connection.


In [ ]:
by_image = defaultdict(list)
for r in formal:
    by_image[r['global_image_index']].append(r)
assert all(len(v)==3 for v in by_image.values())
assert all(len({(r['expected'],r['predicted']) for r in v})==1 for v in by_image.values())
distinct_correct = sum(v[0]['correct'] for v in by_image.values())
formal_correct = sum(r['correct'] for r in formal)
formal_errors = sum(r['error'] for r in formal)
print(f'Distinct accuracy: {distinct_correct}/1000 = {100*distinct_correct/1000:.2f}%')
print(f'Formal repetitions: {formal_correct}/3000 correct; communication errors = {formal_errors}')
print('Block  Images  Correct  Accuracy')
for b in range(0,1000,100):
    count = sum(by_image[i][0]['correct'] for i in range(b,b+100))
    print(f'{b:3d}–{b+99:<3d}   100      {count:2d}     {count}%')
assert distinct_correct==876 and formal_correct==2628


## 3. Request latency and effective throughput

All formal latency observations are included, **including long-tail requests**. The mean and sample standard deviation characterize the 3,000 individual request values; total throughput reflects full SDK run durations and therefore includes between-request controller/software overhead.


In [ ]:
latency_ms = [r['latency_us']/1000 for r in formal]
formal_runs = [r for r in runs if r['phase']=='formal']
total_s = sum(r['run_duration_s'] for r in formal_runs)
throughput = len(formal)/total_s
print(f'Requests: {len(formal):,}')
print(f'Mean latency: {statistics.mean(latency_ms):.3f} ms')
print(f'Median latency: {statistics.median(latency_ms):.3f} ms')
print(f'Sample SD: {statistics.stdev(latency_ms):.3f} ms')
print(f'Min/Max: {min(latency_ms):.3f} / {max(latency_ms):.3f} ms')
print(f'Total application-reported formal time: {total_s:.3f} s')
print(f'Effective throughput: {throughput:.3f} requests/s')
assert round(throughput,3)==14.241


In [ ]:
print('Block       Mean ms  Median ms    SD ms    Min ms    Max ms   images/s')
for b in range(0,1000,100):
    x=[r['latency_us']/1000 for r in formal if r['block_start']==b]
    sec=sum(r['run_duration_s'] for r in formal_runs if r['block_start']==b)
    print(f'{b:3d}–{b+99:<3d} {statistics.mean(x):9.3f} {statistics.median(x):10.3f} {statistics.stdev(x):8.3f} {min(x):9.3f} {max(x):9.3f} {300/sec:10.3f}')


## 4. Validation first requests (not formal benchmark data)

One validation run preceded the three formal runs for each block. The first request after an established connection exhibited a much longer response interval. The measurements do not by themselves identify the root cause or prove a Cloud Run cold start.


In [ ]:
print('Block       First request (ms)    Validation run (s)')
for b in range(0,1000,100):
    first=next(r for r in validation if r['block_start']==b and r['local_image_index']==0)
    run=next(r for r in runs if r['phase']=='validation' and r['block_start']==b)
    print(f'{b:3d}–{b+99:<3d} {first["latency_us"]/1000:18.3f} {run["run_duration_s"]:20.3f}')


## 5. Visual measurement evidence

The photographs and videos in the repository document completion of 100-image blocks, the embedded development environment, power-measurement setup, and selected abnormal connection events. These are supporting evidence rather than substitutes for the source SDK timing records.

### Classification evidence

- [First 100-image result — photograph](emnist_cloud_acc_000.JPG) and [video](emnist_cloud_acc_000_v.mp4)
- Additional block photographs: `emnist_cloud_acc_100.JPG` through `emnist_cloud_acc_900.JPG` at increments of 100
- Additional result videos: `emnist_cloud_acc_100_v.mp4` through `emnist_cloud_acc_900_v.mp4` at increments of 100
- [Example SDK output](emnist_cloud_xilinx_output_example.JPG)

### Network exceptions and power testing

- [Long first request video](emnist_cloud_acc_cold_start_v.mp4) and [TCP error video](emnist_cloud_tcp_error.mp4)
- [Unprogrammed board](emnist_cloud_power_not_programmed.JPG), [USB connected](emnist_cloud_usb.JPG), [USB disconnected](emnist_cloud_no_usb.JPG), and [Continuous inference](emnist_cloud_cont_inference_nousb.JPG)
- [Power with USB video](emnist_cloud_power_usb_v.mp4), [Power without USB video](emnist_cloud_power_nousb_v.mp4)

The power meter measured AC consumption of the local board and attached components; it did not measure energy consumed by Google Cloud Run.


## 6. Vivado reports: live, continuous, and speed configurations

The repository uses the `design_1_wrapper` report naming convention for each configuration. The filenames visible in GitHub screenshots were abbreviated; the notebook locates each file by **report type and configuration suffix**, avoiding a hard-coded guess about truncated stems.

- `design_1_wrapper_utilization_placed_{configuration}.rpt`
- `design_1_wrapper_timing_summary_routed_{configuration}.rpt`
- `design_1_wrapper_power_routed_{configuration}.rpt`

where `{configuration}` is `continuous`, `live`, or `speed`. Reports are interpreted as distinct FPGA builds and should **not** be combined into one resource baseline. Placed utilization is labeled as placed, not post-route.


In [ ]:
from pathlib import Path
import re

report_types = {
    'Placed utilization': 'utilization_placed',
    'Routed timing': 'timing_summary_routed',
    'Routed power': 'power_routed',
}
configs=('continuous','live','speed')

for config in configs:
    print(f'\n=== {config.upper()} CONFIGURATION ===')
    for kind, stem in report_types.items():
        candidates = sorted(Path('.').glob(f'design_1_wrapper*{stem}*{config}*.rpt'))
        if not candidates:
            print(f'  {kind}: not present locally (add report to branch to inspect)')
            continue
        for path in candidates:
            print(f'  {kind}: {path.name}')
            lines=path.read_text(errors='replace').splitlines()
            selectors = {
                'Placed utilization': ('Slice LUTs','Slice Registers','Block RAM Tile','DSPs'),
                'Routed timing': ('WNS','TNS','timing constraints','unconstrained'),
                'Routed power': ('Total On-Chip Power','Dynamic (W)','Device Static (W)','Confidence Level'),
            }[kind]
            shown=0
            for line in lines:
                if any(p.lower() in line.lower() for p in selectors) and line.strip():
                    print('    '+line.strip()[:170]); shown+=1
                    if shown>=10: break
            if shown==0: print('    Inspect report directly; summary terms not matched.')


## 7. Physical power observations

During the EMNIST cloud configuration, the power meter showed approximately **2.9–3.0 W** for the unprogrammed board with Ethernet connected, **4.3–4.5 W** for the programmed board with USB attached, and **4.1–4.3 W** for both programmed idle and continuous inference with USB disconnected. The active and idle ranges overlapped; **no distinct incremental inference power** was resolved by this meter under those conditions. Continuous repeated execution was supported by the observed batch-completion LED activity. These measurements refer only to local embedded-board consumption.

The Vivado power report is a modeled on-chip estimate and must not be substituted for physical meter readings. Energy per request should not be reported as a measured incremental value when the active–idle difference was not resolved.


## 8. Interpretation, limitations, and related branches

**Observed formal result:** 876/1,000 distinct EMNIST Balanced images correct (87.60%); three repeated formal presentations each; 3,000 successful accepted requests; approximately 62.66 ms mean client-observed request latency and 14.24 requests/s by reported batch duration.

**Limitations:** Sample repetition is not independent evidence of accuracy; transient TCP connection failures occurred outside the accepted series; long first-request validation delays were not pooled with steady-state formal data; no Cloud Run-only energy or execution latency was measured; the development TLS build lacked server certificate verification and secure entropy.

The live camera classification proof of concept and associated camera/RTL evidence are documented in the `EMNIST-FPGA-Cloud-Live-Classification-Notebook-and-Data` branch. Source code for this controlled benchmark belongs in the `EMNIST-FPGA-Cloud-Classification-Measurement-Code` branch. The remote Cloud Run service is documented separately.
